# Topic 19c - Model routing for production

Companion notebook for `routing.py`: routing strategies inside a group, the
seven-dimension auto router, session affinity and hosted market routing.

## 0. Import

In [ ]:
from routing import (
    AutoRouter,
    CostBased,
    Deployment,
    LatencyBased,
    LeastBusy,
    MarketOffer,
    RouteChoice,
    RoutingGroup,
    SessionAffinity,
    SimpleShuffle,
    UsageBased,
    heuristic_tier,
    market_route,
    score_prompt,
)

print("imported routing.py")

## 1. Routing strategies inside a group

In [ ]:
a = Deployment("a-azure", "gpt-4.1", provider="azure", cost_per_1k=0.6, weight=1.0)
b = Deployment("b-openai", "gpt-4.1", provider="openai", cost_per_1k=0.4, weight=3.0)
a.record_success(800.0, now=0.0)
b.record_success(120.0, now=0.0)
a.in_flight, b.in_flight = 5, 1

ctx = RouteChoice(rng=lambda: 0.9, now=1.0, usage={"a-azure": 0.9, "b-openai": 0.1})
for strategy in (SimpleShuffle(), LatencyBased(), LeastBusy(), CostBased(), UsageBased()):
    print(f"{strategy.name:24s} ->", strategy.select([a, b], ctx).id)

## 2. A group with cooldowns

A deployment in cooldown is filtered out *before* the strategy runs.

In [ ]:
a.cooldown_until = 0.0
b.cooldown_until = 0.0
group = RoutingGroup("gpt-4.1", [a, b], strategy=LeastBusy())
print("available now:", [d.id for d in group.available(0.0)])

b.record_failure(now=0.0)  # 30s cooldown
print("after b fails:", [d.id for d in group.available(1.0)])
print("picked:", group.pick(RouteChoice(now=1.0)).id)

## 3. Auto routing: score the prompt, pick a tier, say why

In [ ]:
for prompt in (
    "What is the capital of France?",
    "Summarize this api response",
    "Explain the async algorithm then show the sql",
    "Prove it step by step and derive the result",
):
    tier, cause, scores = heuristic_tier(prompt)
    print(f"{tier:9s} [{cause}]  <- {prompt!r}")
print()
print("dimensions:", score_prompt("Write ```code``` with recursion, then prove it?"))

## 4. `model="auto"`: route to a tier, then to a deployment

In [ ]:
groups = {
    "SIMPLE": RoutingGroup("SIMPLE", [Deployment("cheap-1", "gpt-4.1-mini")]),
    "MEDIUM": RoutingGroup("MEDIUM", [Deployment("mid-1", "gpt-4.1")]),
    "COMPLEX": RoutingGroup("COMPLEX", [Deployment("big-1", "gpt-4o")]),
    "REASONING": RoutingGroup("REASONING", [Deployment("o-1", "o4-mini")]),
}
router = AutoRouter(groups, rng=lambda: 0.5)
for prompt in ("What is 2 + 2?", "Summarize this api response", "Prove step by step and derive it"):
    d = router.route(prompt)
    print(f"{d.tier:9s} -> {d.deployment.id:8s} (cause={d.cause})")

## 5. Adaptive exploration (Thompson sampling)

In [ ]:
adaptive = AutoRouter(groups, rng=lambda: 0.5, explore=1.0)
decision = adaptive.route("What is 2 + 2?")
print("tier:", decision.tier, "| cause:", decision.cause)
adaptive.reward(decision.tier, success=True)
print("posterior alpha:", adaptive.alpha[decision.tier], "beta:", adaptive.beta[decision.tier])

## 6. Session affinity (pinned, with fall-through)

In [ ]:
x = Deployment("x", "gpt-4.1")
y = Deployment("y", "gpt-4.1")
affinity_group = RoutingGroup("g", [x, y], strategy=SimpleShuffle(), affinity=SessionAffinity(enabled=True, idle_ttl=3600))

first = affinity_group.pick(RouteChoice(rng=lambda: 0.95), session_id="chat-1")
second = affinity_group.pick(RouteChoice(rng=lambda: 0.0), session_id="chat-1")
print("pinned:", first.id, "->", second.id, "(same regardless of rng)")

y.record_failure(now=0.0)  # cool the pinned deployment down
third = affinity_group.pick(RouteChoice(rng=lambda: 0.0, now=1.0), session_id="chat-1")
print("after y cools down ->", third.id, "(fell through to x, re-pinned)")

## 7. Hosted market-based routing (the contrast)

In [ ]:
offers = [
    MarketOffer("openai/gpt-4o", price_per_1k=5.0, quality=0.90),
    MarketOffer("openai/gpt-4o-mini", price_per_1k=0.5, quality=0.70),
    MarketOffer("anthropic/claude", price_per_1k=3.0, quality=0.95),
]
print("quality:", market_route(offers).model)
print("price  :", market_route(offers, prefer="price").model)
print("filter :", market_route(offers, max_price=1.0, allowed=["openai/*"]).model)
print("exclude:", market_route(offers, excluded=["openai/*"]).model)

## 8. Notes

- A group is the choice unit; the strategy picks a deployment inside it.
- Auto routing is a cheap prior: log `cause=`, audit, and let the bandit correct it.
- Session affinity is a preference: TTL-bounded and it falls through on cooldown.
- Hosted routers trade control for a live market; wildcards keep allow-lists short.

Run the suite: `python -m pytest -q` in this folder.